# Notebook 04: Character-Level Transformer Training (Kaggle GPU Ready)
In this notebook, we build and train a causal Character Transformer directly from scratch in PyTorch.

### Features:
- Kaggle P100/T4 GPU Auto-Detection
- Custom PyTorch Causal Self-Attention (No HF dependency)
- Automatic Mixed Precision (AMP)
- Model Checkpoint saving

In [ ]:
import sys
import os
import zipfile

# Auto-extract zip file if Kaggle uploaded dataset as a raw zip
if os.path.exists("/kaggle/input"):
    for root_dir, dirs, files in os.walk("/kaggle/input"):
        for file in files:
            if file.endswith(".zip") and "wordle" in file.lower():
                zip_path = os.path.join(root_dir, file)
                print(f"Extracting uploaded Kaggle dataset: {zip_path} -> /kaggle/working/")
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall("/kaggle/working/")

# Dynamic path resolution for local and Kaggle environments
possible_roots = [
    ".",
    "..",
    "/kaggle/working",
    "/kaggle/working/wordle",
    "/kaggle/working/wordle-ai",
    "/kaggle/input/wordle",
    "/kaggle/input/wordle/wordle",
    "/kaggle/input/wordle/wordle-ai",
    "/kaggle/input/wordle-ai",
]

# Search recursively inside /kaggle/input for any folder containing 'src'
if os.path.exists("/kaggle/input"):
    for root_dir, dirs, files in os.walk("/kaggle/input"):
        if "src" in dirs:
            possible_roots.insert(0, root_dir)

for root in possible_roots:
    if os.path.exists(os.path.join(root, "src")):
        abs_root = os.path.abspath(root)
        if abs_root not in sys.path:
            sys.path.insert(0, abs_root)
        print(f"Found 'src' package at: {abs_root}")
        break

if os.path.exists("/kaggle/input"):
    print("Running on Kaggle GPU/CPU Environment!")
else:
    print("Running on Local Environment!")
import torch

if os.path.exists("/kaggle/input"):
    !pip install -q pyyaml matplotlib seaborn tqdm

print("PyTorch Version:", torch.__version__, "| CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

from src.wordle.dictionary import WordleDictionary
from src.models.char_transformer import CharTransformerConfig
from src.training.train_char_transformer import train_character_transformer

dict_mgr = WordleDictionary()
config = CharTransformerConfig(d_model=128, n_head=4, num_layers=4, d_ff=256)

model = train_character_transformer(
    config=config,
    dict_mgr=dict_mgr,
    epochs=2,
    batch_size=64,
    num_train_samples=2000,
    num_val_samples=500
)